## @tool装饰器

In [14]:
from langchain_core.tools import tool, StructuredTool
from numba.core.cgutils import Structure
from pydantic import BaseModel, Field


class FieldInfo(BaseModel):
    a: int = Field(description='参数1')
    b: int = Field(description='参数2')


'''
    函数描述信息必须填写
    LLM 会利用函数名和 docstring 来决定何时调用
'''


@tool(name_or_callable='add_two_sum', description='add two',
      return_direct=True, args_schema=FieldInfo)
def add_number(a: int, b: int) -> int:
    """计算两个整数的和"""
    return a + b


print(f"name={add_number.name}")  #默认是函数的名称
print(f"args={add_number.args}")  #参数描述
print(f"description={add_number.description}")  #默认是函数的说明信息
print(f"return_direct={add_number.return_direct}")  #默认值是False

name=add_two_sum
args={'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
description=add two
return_direct=True


`举例二`：进阶

In [14]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain.agents import create_agent
from langchain_core.tools import tool
import os
import dotenv

'''
create_react_agent已经作废，换成create_agent
'''
dotenv.load_dotenv(dotenv_path='../chapter02-model IO/.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
llm = ChatOpenAI(
    model='qwen3-vl-flash-2026-01-22',

    temperature=0.6
)


@tool
def calculate_multiply(a: int, b: int) -> int:
    """计算两个数字的乘积。当需要数学运算时使用此工具。"""
    return a * b


# 定义工具列表
tools = [calculate_multiply]

# 创建一个自动执行循环的 Agent
# 它会自动解析 -> 执行工具 -> 获取结果 -> 继续下一步
agent_executor = create_agent(model=llm, tools=tools)

# 执行
result = agent_executor.invoke({"messages": [("human", "42 * 10+1 是多少？")]})

print(result["messages"][-1].content)
# 输出: 42 * 10 是 420。

42 * 10 + 1 = 420 + 1 = 421。


## StructuredTool的from_function()

In [11]:
'''
StructuredTool 是 LangChain 中定义复杂工具的底层类，它允许你通过定义参数的 JSON Schema，强制 LLM 严格遵守输入参数的格式。
虽然 @tool 装饰器是创建工具的最常用方法（它底层就是调用的 StructuredTool），但在某些需要动态生成工具、或需要通过类方法精细控制的场景下，
直接使用 StructuredTool 是更健壮的选择。
'''

# TODO 备注
'''
    什么时候用它？
    当你需要参数约束：比如参数必须在一定范围内，或者是特定的枚举值。
    当函数不是你写的：你不能通过装饰器修改外部函数的代码。
    需要动态工具：需要在运行时动态创建工具逻辑。
'''
# 举例1
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field


class FieldInfo(BaseModel):
    query:str=Field(description='要搜索的关键字')

# 1. 定义一个简单的处理函数
def multiply(query:str) -> str:
    """查询baidu的结果"""
    return '查询的结果'


# 2. 使用 StructuredTool 手动定义 Schema
# 这比直接写 @tool 有更强的灵活性，比如手动指定参数描述
multiply_tool = StructuredTool.from_function(
    func=multiply,
    name="multiply_tool",
    description="用于执行查询网页的工具",
    # True当工具调用之后直接把结果返回给用户，False则把决定权交给Agent，默认False
    # return_direct=True
    # 你甚至可以手动定义 args_schema 来精细控制参数校验
    args_schema=FieldInfo
)
print(f'name={multiply_tool.name}')
print(f'args={multiply_tool.args}')
print(f'description={multiply_tool.description}')
print(f'return_direct={multiply_tool.return_direct}')
# 使用
print(multiply_tool.invoke({'query':'明天天气如何'}))

name=multiply_tool
args={'query': {'description': '要搜索的关键字', 'title': 'Query', 'type': 'string'}}
description=用于执行查询网页的工具
return_direct=False
查询的结果
